# 实验 06：TextCNN 情感分类
使用不同宽度的卷积核提取局部词组特征，再进行最大池化。
阅读：[对应教材章节](https://zh.d2l.ai/chapter_natural-language-processing-applications/sentiment-analysis-cnn.html)。
全部文本实验使用 IMDB 英文影评，不更换数据集，不额外下载预训练词向量。


In [ ]:
from pathlib import Path
import os, sys

# 从项目根目录、notebooks/ 或它们的子目录启动时都能找到 labkit。
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "labkit").is_dir()), None)
if ROOT is None:
    raise RuntimeError("请将整个课程目录解压后，从项目根目录启动 JupyterLab")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("项目根目录:", ROOT)


## 1. 配置
正式实验使用完整数据，`SMOKE=True` 仅检查流程。


In [ ]:
from datetime import datetime
import inspect
import torch
from labkit.common import choose_device, seed_everything
from labkit.data import load_text_data, tokenize
from labkit.text_models import build_text_model
from labkit.image_models import parameter_count
from labkit.engine import run_epoch, fit, final_test
from labkit.plots import plot_history, plot_confusion, save_run_plots
from labkit.inference import predict_text

MODEL = "textcnn"
SMOKE = os.environ.get("LAB_SMOKE", "0") == "1"
SEED, BATCH_SIZE = 42, 64
MAX_LENGTH, MAX_VOCAB, MIN_FREQ = 256, 20000, 2
EPOCHS, LR = (1 if SMOKE else 8), 0.001
DATA_DIR = ROOT / "data"
RUN_DIR = ROOT / "runs" / (MODEL + datetime.now().strftime("_%Y%m%d_%H%M%S_%f"))
seed_everything(SEED)
device = choose_device("auto")
print("device:", device, "SMOKE:", SMOKE, "结果目录:", RUN_DIR)


## 2. 原始影评、分词、词表与 DataLoader
标签 0=negative、1=positive。词表只统计训练集，`<pad>=0`、`<unk>=1`。
序列先截断至最大长度，再按当前批次动态填充。文本 CNN 至少需要 5 个位置。


In [ ]:
limits = [128, 64, 64] if SMOKE else [0, 0, 0]
bundle = load_text_data(DATA_DIR, BATCH_SIZE, SEED, workers=0,
                        max_length=MAX_LENGTH, max_vocab=MAX_VOCAB, min_freq=MIN_FREQ,
                        max_train=limits[0], max_val=limits[1], max_test=limits[2])
print("规模:", bundle.meta["sizes"], "词表大小:", len(bundle.vocab))
sample = bundle.train.dataset.records[0]
print("影评片段:", sample["text"][:350])
print("标签:", sample["label"])
print("分词:", tokenize(sample["text"])[:25])
tokens, lengths, labels = next(iter(bundle.val))
print("tokens:", tokens.shape, "真实长度:", lengths[:8], "labels:", labels[:8])
print("特殊 token:", bundle.vocab.tokens[:2])


## 3. 模型结构
词向量通过当前训练集学习。输出为两个类别的 logits，交叉熵之前不做 softmax。


In [ ]:
model = build_text_model(MODEL, len(bundle.vocab)).to(device)
print(model)
print("参数量:", parameter_count(model))
print(inspect.getsource(type(model)))
model.eval()
with torch.no_grad():
    logits = model(tokens[:2].to(device), lengths[:2].to(device))
print("输出形状:", logits.shape)


## 4. 训练与验证
阅读完整循环，注意验证阶段没有反向传播，梯度裁剪可以限制过大的梯度。


In [ ]:
print(inspect.getsource(run_epoch))
print(inspect.getsource(fit))


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
config = {"task": "text", "model": MODEL, "seed": SEED,
          "batch_size": BATCH_SIZE, "max_length": MAX_LENGTH,
          "max_vocab": MAX_VOCAB, "min_freq": MIN_FREQ,
          "epochs": EPOCHS, "lr": LR, "smoke": SMOKE,
          "parameters": parameter_count(model), "torch_version": str(torch.__version__)}
history = fit(model, bundle, optimizer, device, RUN_DIR, config,
               epochs=EPOCHS, amp=False, grad_clip=1.0)
plot_history(history)


## 5. 最终测试
恢复最佳验证模型后，报告 accuracy、macro F1 和混淆矩阵。


In [ ]:
metrics = final_test(model, bundle, device, RUN_DIR)
save_run_plots(history, metrics, RUN_DIR)
plot_confusion(metrics["confusion_matrix"], bundle.meta["classes"])
print("accuracy:", metrics["accuracy"], "macro F1:", metrics["macro_f1"])


## 6. 从磁盘恢复模型并预测
预测必须复用保存的词表和分词规则，不能临时创建新词表。低置信度或错误预测也是需要分析的结果。


In [ ]:
examples = ["The story was moving and the acting was excellent.",
            "The movie was boring and I would not recommend it.",
            "The acting was great, but the plot was disappointing."]
for text in examples:
    print(text)
    print(predict_text(RUN_DIR, text, str(device)))


## 7. 实操练习
1. 将卷积核改为只使用 (3,)，比较参数量与验证效果。
2. 解释 TextCNN 为什么需要处理 PAD 对池化的影响。
3. 与 BiLSTM 比较 macro F1、训练耗时和错误样本，避免只按单一指标判断。

与另一种文本模型比较时，保持种子、词表容量、最大长度、训练轮数一致。结果使用 `compare_runs.py` 汇总。
